# 3回目: ASR + LLM で音声認識精度を上げる (Colab)

GPU ランタイム推奨 (T4 以上)。`ランタイム → ランタイムのタイプを変更 → T4 GPU`。

流れ:
1. セットアップ
2. データアップロード (`data/sample.mp4`)
3. 実践① WhisperX で文字起こし
4. 実践② LLM で補正
5. セルフ1 検証ループ
6. セルフ2 精度向上

## 1. セットアップ

In [ ]:
!pip install -q whisperx openai python-dotenv jiwer

In [ ]:
import os
os.environ['LLM_URL']     = 'https://www.sunyai.com/litellm/'
os.environ['LLM_MODEL']   = 'gpt-5-mini'
os.environ['LLM_API_KEY'] = 'sk-...'  # 自分のキーに置き換え

## 2. データアップロード
`sample.mp4` を左のファイルペインにアップロードして `/content/sample.mp4` に置く。

In [ ]:
AUDIO = '/content/sample.mp4'
assert os.path.exists(AUDIO), 'sample.mp4 をアップロードしてください'

## 3. 実践① WhisperX で文字起こし

In [ ]:
import torch, whisperx, json
device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
model = whisperx.load_model('large-v3', device=device, compute_type=compute_type)
audio = whisperx.load_audio(AUDIO)
result = model.transcribe(audio, batch_size=8)
lang = result['language']
align_model, meta = whisperx.load_align_model(language_code=lang, device=device)
aligned = whisperx.align(result['segments'], align_model, meta, audio, device=device, return_char_alignments=False)
raw_text = ' '.join(s['text'].strip() for s in aligned['segments'])
print('language:', lang)
print(raw_text[:400])
with open('/content/01_asr_raw.txt','w') as f: f.write(raw_text)
with open('/content/01_asr_raw.json','w') as f: json.dump({'language': lang, 'segments': aligned['segments']}, f, ensure_ascii=False, indent=2)

## 4. 実践② LLM で補正

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=os.environ['LLM_URL'], api_key=os.environ['LLM_API_KEY'])
SYS = '''あなたは ASR 後処理専門の校正エンジンです。
[絶対ルール]
1. 新しい単語を追加してはいけません。
2. 既存単語の置換・削除のみ許可。
3. 判断不能なら原文維持。
4. 出力は補正後のテキストのみ。'''
resp = client.chat.completions.create(model=os.environ['LLM_MODEL'], temperature=0.0, messages=[
    {'role':'system','content':SYS},
    {'role':'user','content':raw_text},
])
corrected = resp.choices[0].message.content.strip()
print(corrected[:400])
with open('/content/02_asr_llm_corrected.txt','w') as f: f.write(corrected)

In [ ]:
import difflib
diff = '\n'.join(difflib.unified_diff(raw_text.splitlines(), corrected.splitlines(), fromfile='raw', tofile='corrected', lineterm=''))
print(diff[:2000])

## 5. セルフ1 検証ループ
LLM が追加した単語が音声に存在するか WhisperX alignment で確認。

In [ ]:
import re
def tokenize(t): return re.findall(r'[A-Za-z]+|[一-龥ぁ-んァ-ヶ]+|\d+', t)
added = [w for w in tokenize(corrected) if w not in set(tokenize(raw_text))]
print('naive added:', added[:30])
# 修正後テキストの forced alignment
fake = [{'start':0.0,'end':len(audio)/16000,'text':corrected}]
checked = whisperx.align(fake, align_model, meta, audio, device=device, return_char_alignments=False)
words = [w for s in checked['segments'] for w in (s.get('words') or [])]
suspicious = [w for w in words if w.get('word','').strip() in set(added)]
for s in suspicious[:30]:
    print(s)

## 6. セルフ2 精度向上 (WER)
手修正した reference.txt をアップロードしてから実行。

In [ ]:
from jiwer import wer
REF = '/content/reference.txt'
if os.path.exists(REF):
    ref = open(REF).read()
    print('WER raw :', wer(ref, raw_text))
    print('WER llm :', wer(ref, corrected))
else:
    print('reference.txt をアップロードしてください')